In [ ]:
# Cell 1: Install all required libraries

!pip install -q transformers datasets peft accelerate gradio detoxify

In [ ]:
# Cell 2: Import all necessary libraries
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import gradio as gr
from detoxify import Detoxify
import time

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
# Cell 3: Mount Google Drive to save our model permanently
from google.colab import drive
drive.mount('/content/drive')

# Create a directory for Week 5 models
import os
model_save_path = '/content/drive/MyDrive/Capstone_Week5_Models'
os.makedirs(model_save_path, exist_ok=True)

print(f" Google Drive mounted successfully!")
print(f" Models will be saved to: {model_save_path}")

In [ ]:
# Cell 4: Load the base GPT-2 model and tokenizer
print("Loading GPT-2 Small model and tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    device_map="auto",
    torch_dtype=torch.float16
)

print(f" Model loaded: GPT-2 Small")
print(f" Total parameters: {base_model.num_parameters():,}")
print(f" Model is on: {base_model.device}")


In [ ]:
# Cell 5: Configure LoRA with our optimal hyperparameters from Week 4
print("Configuring LoRA with optimal settings from Week 4...")

# LoRA configuration - using best parameters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # Rank = 8 (best from Week 4)
    lora_alpha=16,          # Alpha = 16 (best from Week 4)
    lora_dropout=0.1,
    target_modules=["c_attn"],  # Apply LoRA to attention layers
    bias="none"
)

# Apply LoRA to the base model
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print(f"\n LoRA configuration complete!")
print(f" Using optimal settings: r=8, alpha=16")

In [ ]:
# Cell 6: Load WikiText-103 dataset for fine-tuning
print("Loading WikiText-103 dataset...")

# Load dataset
dataset = load_dataset("wikitext", "wikitext-103-v1")

# Using small subsets for quick training
train_dataset = dataset["train"].select(range(10000))  # 10k samples
val_dataset = dataset["validation"].select(range(1000))  # 1k samples

print(f"✓ Training samples: {len(train_dataset):,}")
print(f"✓ Validation samples: {len(val_dataset):,}")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

# Tokenize datasets
print("\nTokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

print(" Tokenization complete!")

In [ ]:
# Cell 7: Set up training configuration with optimal hyperparameters
print("Configuring training parameters...")

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments - using  optimal settings from Week 4
training_args = TrainingArguments(
    output_dir="./week5_lora_training",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size = 16
    learning_rate=3e-5,              # optimal learning rate
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True,                       # mixed precision for speed
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=1,              # Only keep best checkpoint
    report_to="none",
)

print("Training configuration complete!")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f" Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
# Cell 8: Train the LoRA model
print("="*70)
print("STARTING TRAINING")
print("="*70)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

# Start training
print("\nTraining in progress...")
train_result = trainer.train()

# Evaluate final model
print("\n" + "="*70)
print("EVALUATING FINAL MODEL")
print("="*70)
eval_results = trainer.evaluate()

# Calculate perplexity
final_loss = eval_results['eval_loss']
perplexity = np.exp(final_loss)

print(f"\n Training complete!")
print(f"Final validation loss: {final_loss:.4f}")
print(f"Final perplexity: {perplexity:.2f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")

In [ ]:
# Cell 9: Save the fine-tuned model to Google Drive
print("Saving model to Google Drive...")

# Save LoRA adapter weights
final_model_path = f"{model_save_path}/lora_gpt2_week5_final"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f" Model saved successfully!")
print(f" Location: {final_model_path}")
print(f"\nModel will persist in your Google Drive even after this session ends.")

# Verify files were saved
import os
saved_files = os.listdir(final_model_path)
print(f"\n Saved files: {saved_files}")

In [ ]:
# Cell 10: Load toxicity detection model for safety filtering
print("Loading toxicity detection model...")

# Initialize Detoxify
toxicity_detector = Detoxify('original')

print("Toxicity detector loaded!")
print("\nThis will help us filter harmful content in generated text.")

# Test the toxicity detector
test_texts = [
    "This is a normal, friendly sentence.",
    "I hate you and wish terrible things upon you."
]

print("\nTesting toxicity detector:")
for text in test_texts:
    results = toxicity_detector.predict(text)
    toxicity_score = results['toxicity']
    print(f"  Text: '{text}'")
    print(f"  Toxicity score: {toxicity_score:.4f} {' HIGH' if toxicity_score > 0.5 else '✓ Safe'}\n")

In [ ]:
# Cell 11: Define the text generation function with safety checks
def generate_text(prompt, temperature=0.9, top_p=0.9, max_length=100, num_return_sequences=1):
    """
    Generate text using the fine-tuned model with toxicity filtering.

    Parameters:
    - prompt: Input text to continue
    - temperature: Controls randomness (0.7-1.2 recommended)
    - top_p: Nucleus sampling parameter (0.9-0.95 recommended)
    - max_length: Maximum tokens to generate
    - num_return_sequences: Number of different outputs to generate
    """

    # Record start time for latency measurement
    start_time = time.time()

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1  # Prevent repetition
        )

    # Decode generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    # Calculate generation time
    generation_time = time.time() - start_time

    # Check toxicity for each generated text
    results = []
    for text in generated_texts:
        toxicity_scores = toxicity_detector.predict(text)
        toxicity_score = toxicity_scores['toxicity']

        results.append({
            'text': text,
            'toxicity': toxicity_score,
            'is_safe': toxicity_score < 0.5
        })

    return results, generation_time

# Test the generation function
print("Testing text generation function...\n")
test_prompt = "Artificial intelligence is transforming"
test_results, test_time = generate_text(test_prompt, temperature=0.9, top_p=0.9, max_length=50)

print(f"Prompt: '{test_prompt}'")
print(f"\nGenerated text:")
print(test_results[0]['text'])
print(f"\nToxicity score: {test_results[0]['toxicity']:.4f} {'✓ Safe' if test_results[0]['is_safe'] else '⚠️ Warning'}")
print(f"Generation time: {test_time:.2f} seconds")

In [ ]:
# Cell 12: Build the Gradio interface
def gradio_generate(prompt, temperature, top_p, max_length):
    """
    Wrapper function for Gradio interface.
    """
    if not prompt.strip():
        return "⚠️ Please enter a prompt!", "", ""

    # Generate text
    results, gen_time = generate_text(
        prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_length=max_length,
        num_return_sequences=1
    )

    result = results[0]

    # Format output
    generated_text = result['text']
    toxicity_score = result['toxicity']
    is_safe = result['is_safe']

    # Create safety status message
    if is_safe:
        safety_msg = f" Safe (Toxicity: {toxicity_score:.4f})"
    else:
        safety_msg = f" Warning: Potentially toxic content (Score: {toxicity_score:.4f})"

    # Create metadata
    metadata = f"""**Generation Statistics:**
- Generation time: {gen_time:.2f} seconds
- Tokens generated: ~{len(generated_text.split())} words
- Temperature: {temperature}
- Top-p: {top_p}
- Max length: {max_length} tokens"""

    return generated_text, safety_msg, metadata


# Create Gradio interface
print("Creating Gradio interface...\n")

demo = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            label="Enter your prompt",
            placeholder="Type your text here... e.g., 'The future of technology is'",
            lines=3
        ),
        gr.Slider(
            minimum=0.5,
            maximum=1.5,
            value=0.9,
            step=0.1,
            label="Temperature (controls randomness)"
        ),
        gr.Slider(
            minimum=0.8,
            maximum=1.0,
            value=0.9,
            step=0.05,
            label="Top-p / Nucleus Sampling (controls diversity)"
        ),
        gr.Slider(
            minimum=50,
            maximum=200,
            value=100,
            step=10,
            label="Max Length (tokens)"
        )
    ],
    outputs=[
        gr.Textbox(label="Generated Text", lines=8),
        gr.Textbox(label="Safety Check"),
        gr.Markdown(label="Generation Metadata")
    ],
    title=" Week 5: LLM Deployment Interface",
    description="""**Fine-tuned GPT-2 with LoRA** (r=8, alpha=16, perplexity=33.66)

This interface demonstrates:
- ✅ Parameter-efficient fine-tuning (LoRA)
- ✅ Optimal sampling strategies (Nucleus sampling)
- ✅ Safety filtering (Toxicity detection)
- ✅ Real-time inference with latency monitoring

**Instructions:** Enter a prompt, adjust parameters, and click Submit!""",
    examples=[
        ["Artificial intelligence is transforming", 0.9, 0.9, 100],
        ["The future of medicine will include", 0.8, 0.95, 100],
        ["In the field of robotics,", 1.0, 0.9, 80]
    ],
    theme=gr.themes.Soft()
)

print("✓ Gradio interface created!")
print("\nReady to launch!")

In [ ]:
# Cell 13: Launch the Gradio interface
print("Launching Gradio interface...")
print("="*70)

# Launch with public sharing enabled
demo.launch(share=True, debug=True)

print("\n Interface is now running!")
print(" You can interact with it above")
print(" The public link allows you to share with others (optional)")

In [ ]:
# Cell 14: Collect deployment performance measurements
print("="*70)
print("COLLECTING DEPLOYMENT MEASUREMENTS FOR WEEK 5 REPORT")
print("="*70)

# Test multiple prompts to get average metrics
test_prompts = [
    "Artificial intelligence is transforming",
    "The future of medicine will include",
    "In the field of robotics,",
    "Machine learning algorithms can",
    "Natural language processing enables"
]

generation_times = []
toxicity_scores = []
word_counts = []

print("\nRunning 5 test generations...\n")

for prompt in test_prompts:
    results, gen_time = generate_text(prompt, temperature=0.9, top_p=0.9, max_length=100)

    generation_times.append(gen_time)
    toxicity_scores.append(results[0]['toxicity'])
    word_counts.append(len(results[0]['text'].split()))

    print(f"✓ Prompt: '{prompt[:40]}...'")
    print(f"  Time: {gen_time:.2f}s | Toxicity: {results[0]['toxicity']:.4f} | Words: {len(results[0]['text'].split())}\n")

# Calculate statistics
avg_time = np.mean(generation_times)
avg_toxicity = np.mean(toxicity_scores)
avg_words = np.mean(word_counts)

print("="*70)
print("DEPLOYMENT METRICS SUMMARY")
print("="*70)
print(f"\n Performance Metrics:")
print(f"   • Average generation time: {avg_time:.2f} seconds")
print(f"   • Min/Max time: {min(generation_times):.2f}s / {max(generation_times):.2f}s")
print(f"   • Average words generated: {avg_words:.1f}")
print(f"\n Safety Metrics:")
print(f"   • Average toxicity score: {avg_toxicity:.4f}")
print(f"   • All generations safe: {all(score < 0.5 for score in toxicity_scores)}")
print(f"\n Model Specifications:")
print(f"   • LoRA trainable parameters: 294,912 (0.24% of base model)")
print(f"   • Model size on disk: ~1-2 MB (LoRA adapter only)")
print(f"   • Base model: GPT-2 Small (124M parameters)")
print(f"   • Fine-tuning dataset: WikiText-103 (10k samples)")
print(f"   • Final perplexity: 33.66")
print(f"\n Optimal Configuration:")
print(f"   • LoRA rank (r): 8")
print(f"   • LoRA alpha: 16")
print(f"   • Learning rate: 3e-5")
print(f"   • Sampling: Nucleus (top-p=0.9)")
print(f"   • Temperature: 0.9")
print(f"\n Hardware Used:")
print(f"   • GPU: {torch.cuda.get_device_name(0)}")
print(f"   • Training time: ~17 minutes (3 epochs)")
print(f"   • Inference device: GPU (CUDA)")

print("\n" + "="*70)
print("✓ All measurements collected for Week 5 report!")
print("="*70)